---

## CropGuard: AI - Powered Tomato Disease Detection Model

## Why Crop Disease Detection Matters

**The Challenge**: Crop diseases cause up to 40% of global crop losses annually, threatening food security for millions of people.<br> 
Early detection is critical but requires expert knowledge that may not be available in remote or resource-constrained regions where WFP operates.

**The Solution**: Deep learning-powered image classification can democratize disease detection, enabling:
- **Farmers** to identify diseases using smartphone cameras
- **Agricultural extension workers** to diagnose problems in the field
- **WFP field officers** to assess crop health rapidly during assessments
- **Early intervention** before diseases spread across entire fields

**This Lab**: We'll build a CNN to classify tomato leaf diseases from images. 
Tomatoes are a critical crop in many food-insecure regions, and diseases like early blight, late blight, and leaf mold can devastate harvests.

### What Makes This Different from General Image Classification?

Agricultural disease detection has unique challenges:
- **Similar visual symptoms** across different diseases
- **Variable image quality** from field conditions (lighting, angle, focus)
- **Class imbalance** (healthy leaves are more common than diseased)
- **Need for high confidence** (false negatives = crop loss; false positives = unnecessary treatment)
- **Mobile deployment** required for field use


### What is CropGuard?
CropGuard AI is an end-to-end deep learning image classication system designed to help smallholder farmers in Ghana detect tomato leaf diseases instantly from a photo without no complex setup required and can be used with or without any training.

The app classifies a tomato leaf image into one of 10 categories (9 diseases + healthy) and immediately recommends a treatment action, helping farmers intervene early before significant crop loss occurs. It is designed for use by tomato farmers across all the region of Ghana

## Project Framework

### 1. Data Acquisition & Preprocessing
*   **Source**: PlantVillage dataset (18,000+ tomato leaf images, 10 classes).
*   **Initial Analysis**: Class distribution, imbalance ratio (14:1 most vs. least common), image size check.
*   **Augmentation**: Rotation, flip, zoom, brightness, fill (to enhance robustness and mitigate overfitting).
*   **Normalization**: Preprocessing function (e.g., `preprocess_input` for ResNet50) to scale pixel values.

### 2. Model Architecture (Transfer Learning)
*   **Base Model**: ResNet50 (pre-trained on ImageNet).
    *   **Strategy**: Weights frozen to leverage pre-learned features.
*   **Custom Classification Head**:
    *   `GlobalAveragePooling2D`: Reduces spatial dimensions.
    *   `BatchNormalization`: Stabilizes training.
    *   `Dense` (256 units, ReLU activation): Learns high-level features.
    *   `Dropout` (0.4 rate): Prevents overfitting.
    *   `Dense` (10 units, Softmax activation): Outputs class probabilities for 10 tomato categories.

### 3. Model Training & Optimization
*   **Loss Function**: Categorical Crossentropy (suitable for multi-class classification).
*   **Optimizer**: Adam (adaptive learning rate optimization).
*   **Metrics**: Accuracy (to evaluate classification performance).
*   **Class Weighting**: Balanced weights applied to address dataset imbalance.
* 

### 4. Evaluation
*   **Metrics**: Validation Accuracy, Loss.
*   **Advanced Metrics**: Classification Report, Confusion Matrix (for detailed performance analysis per class).

### 5. Deployment (Streamlit Web Application)
*   **Interface**: User-friendly web application.
*   **Input**: Accepts a tomato leaf image from the user by upload or capture.
*   **Output**: Provides predicted disease class, confidence score, and recommended treatment actions.

### 6. Expected Outcome
*   A deployable disease detection tool with high validation accuracy across all 10 tomato classes.
*   An accessible and intuitive user interface for field use by farmers.

#  Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# drive.mount("/content/drive", force_remount=True)

#  Imports Libraries and Dependencies

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Paths and constants

In [ ]:
TRAIN_DIR  = '/content/drive/MyDrive/cropguard/tomato_dataset/train'
VALID_DIR  = '/content/drive/MyDrive/cropguard/tomato_dataset/valid'
MODEL_DIR  = '/content/drive/MyDrive/cropguard/models'
IMG_SIZE   = 224
BATCH_SIZE = 32
SEED       = 42

os.makedirs(MODEL_DIR, exist_ok=True)

# EDA

In [ ]:
class_names = sorted(os.listdir(TRAIN_DIR))
counts = [len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in class_names]
clean_names = [c.replace('Tomato___', '').replace('_', ' ') for c in class_names]

plt.figure(figsize=(12, 5))
bars = plt.barh(clean_names, counts, color='steelblue')
plt.xlabel('Number of Images')
plt.title('Training Set — Class Distribution')
for bar, count in zip(bars, counts):
    plt.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
             str(count), va='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f"Total training images : {sum(counts)}")
print(f"Most common class     : {clean_names[counts.index(max(counts))]} ({max(counts)})")
print(f"Least common class    : {clean_names[counts.index(min(counts))]} ({min(counts)})")
print(f"Imbalance ratio       : {max(counts)/min(counts):.1f}:1")

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
axes = axes.flatten()
for idx, (cls, name) in enumerate(zip(class_names, clean_names)):
    cls_path = os.path.join(TRAIN_DIR, cls)
    img = Image.open(os.path.join(cls_path, os.listdir(cls_path)[0])).convert('RGB')
    axes[idx].imshow(img)
    axes[idx].set_title(name, fontsize=9)
    axes[idx].axis('off')
plt.suptitle('Sample Image — Each Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Data preprocessing pipeline

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=40,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.3,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)
valid_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=True, seed=SEED
)
valid_gen = valid_datagen.flow_from_directory(
    VALID_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=False
)

NUM_CLASSES = len(train_gen.class_indices)
print(f"Number of classes: {NUM_CLASSES}")

class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
class_weight_dict = dict(enumerate(class_weights))

# Model Architecture

In [ ]:
base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False

inputs = Input(shape=(224, 224, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs, name='CropGuard_ResNet50')
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Model Training

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint(f'{MODEL_DIR}/best_model.keras', monitor='val_accuracy', save_best_only=True)
]

history = model.fit(
    train_gen,
    validation_data=valid_gen,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=callbacks
)


# Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

valid_gen.reset()
preds = model.predict(valid_gen)
pred_classes = np.argmax(preds, axis=1)
true_classes = valid_gen.classes
class_labels = [c.replace('Tomato___', '').replace('_', ' ')
                for c in list(valid_gen.class_indices.keys())]

print(classification_report(true_classes, pred_classes, target_names=class_labels))

cm = confusion_matrix(true_classes, pred_classes)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix — Tomato Disease Detection')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# Save Model

In [ ]:
# Save Model
model.save(f'{MODEL_DIR}/cropguard_resnet50.keras')
print("Model saved.")

# Load the best model for inference

In [ ]:
# Load the best model
from tensorflow.keras.models import load_model

best_model = load_model(f'{MODEL_DIR}/best_model.keras')
print("Best model loaded successfully.")

# Evaluation Report Summary



1.   94% overall accuracy is a strong result for a 10-class problem
2.   Healthy class is the best performing at 0.98 F1 — the model is very confident when a leaf is healthy
Yellow leaf curl virus hit perfect precision (1.00) — zero false positives

1.   Spider mites has the lowest precision (0.79) meaning the model sometimes mistakes other diseases for it — worth flagging in your report
2.   Target spot and Early blight are the weakest overall, which makes sense as they share similar visual symptoms




**Strengths**

1. Healthy, Mosaic virus, and Spider mites are almost perfectly classified — very few off-diagonal values

2. Yellow Leaf Curl Virus (991/1071) is the most dominant class and the model handles it confidently with zero confusion with healthy leaves

3. Leaf Mold and Septoria leaf spot are clean with minimal cross-confusion

**Weaknesses**

1. Bacterial spot → Target Spot (26 misclassified) — the biggest off-diagonal value in the matrix. Both diseases produce dark lesions on leaves, so visual similarity is the likely cause

2. Yellow Leaf Curl Virus → Spider mites (61 misclassified) — both cause leaf curling and distortion, which explains the confusion

3. Target Spot → Spider mites (20 misclassified) — similar spotting patterns

4. Late blight → Early blight (11 misclassified) — expected, both are blight diseases with overlapping visual symptoms